# CartPole DQN Code

In [7]:
!pip install gymnasium tensorflow numpy

In [9]:
import gymnasium as gym
import numpy as np
import tensorflow as tf
from collections import deque
import random

# Hyperparameters
EPISODES = 10
BATCH_SIZE = 32
GAMMA = 0.99
EPSILON = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995
LEARNING_RATE = 0.001

# Build the Neural Network Model
def build_model(state_size, action_size):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(24, input_dim=state_size, activation='relu'),
        tf.keras.layers.Dense(24, activation='relu'),
        tf.keras.layers.Dense(action_size, activation='linear')
    ])
    model.compile(loss='mse', optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE))
    return model

# Replay Memory
memory = deque(maxlen=2000)

# Environment Setup
env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0]
action_size = env.action_space.n
model = build_model(state_size, action_size)
target_model = build_model(state_size, action_size)

# Update target model to match the main model
def update_target_model():
    target_model.set_weights(model.get_weights())

# Choose action using epsilon-greedy policy
def get_action(state, epsilon):
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)
    q_values = model.predict(state, verbose=0)
    return np.argmax(q_values[0])

# Train the model with experience replay
def train_model():
    if len(memory) < BATCH_SIZE:
        return
    minibatch = random.sample(memory, BATCH_SIZE)
    states = np.array([t[0].reshape(state_size) for t in minibatch])
    actions = np.array([t[1] for t in minibatch])
    rewards = np.array([t[2] for t in minibatch])
    next_states = np.array([t[3].reshape(state_size) for t in minibatch])
    dones = np.array([t[4] for t in minibatch])

    targets = model.predict(states, verbose=0)
    next_q_values = target_model.predict(next_states, verbose=0)
    for i in range(BATCH_SIZE):
        if dones[i]:
            targets[i][actions[i]] = rewards[i]
        else:
            targets[i][actions[i]] = rewards[i] + GAMMA * np.max(next_q_values[i])

    model.fit(states, targets, epochs=1, verbose=0)

# Main Training Loop
epsilon = EPSILON
for episode in range(EPISODES):
    state = env.reset()
    # Check if the state is a tuple and extract the first element if necessary
    if isinstance(state, tuple):
        state = state[0]
    state = np.array(state).reshape(1, state_size)
    total_reward = 0
    done = False

    while not done:
        action = get_action(state, epsilon)
        next_state, reward, done, _, _ = env.step(action)
        # Check if the next_state is a tuple and extract the first element if necessary
        if isinstance(next_state, tuple):
            next_state = next_state[0]
        next_state = np.array(next_state).reshape(1, state_size)

        # Store experience in memory
        memory.append((state, action, reward, next_state, done))
        state = next_state

        # Train the model
        train_model()

    # Update target model every episode
    update_target_model()

    # Decay epsilon
    if epsilon > EPSILON_MIN:
        epsilon *= EPSILON_DECAY

    print(f"Episode: {episode + 1}, Total Reward: {total_reward}, Epsilon: {epsilon:.3f}")

env.close()

Episode: 1, Total Reward: 0, Epsilon: 0.995
Episode: 2, Total Reward: 0, Epsilon: 0.990
Episode: 3, Total Reward: 0, Epsilon: 0.985
Episode: 4, Total Reward: 0, Epsilon: 0.980
Episode: 5, Total Reward: 0, Epsilon: 0.975
Episode: 6, Total Reward: 0, Epsilon: 0.970
Episode: 7, Total Reward: 0, Epsilon: 0.966
Episode: 8, Total Reward: 0, Epsilon: 0.961
Episode: 9, Total Reward: 0, Epsilon: 0.956
Episode: 10, Total Reward: 0, Epsilon: 0.951
